# **BHS Final Project 2026**

## **EEG Data Preprocessing**

### Import Library 

In [1]:
import mne
import gc
import os
import psutil
import sys
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

os.makedirs("EEG_plots", exist_ok=True) #create a folder to save plots

### Check RAM before start

In [2]:
# RAM checking function

def check_ram(label=""):
    """Print current RAM usage."""
    process = psutil.Process(os.getpid())
    used_mb  = process.memory_info().rss / 1024 ** 2
    total_mb = psutil.virtual_memory().total / 1024 ** 2
    avail_mb = psutil.virtual_memory().available / 1024 ** 2
    pct      = used_mb / total_mb * 100
 
    bar_len  = 30
    filled   = int(bar_len * used_mb / total_mb)
    bar      = "█" * filled + "░" * (bar_len - filled)
 
    tag = f"[{label}] " if label else ""
    print(f"\n{tag}RAM Usage")
    print(f"  [{bar}]")
    print(f"  This notebook : {used_mb:>7.0f} MB")
    print(f"  Available     : {avail_mb:>7.0f} MB")
    print(f"  Total system  : {total_mb:>7.0f} MB  ({pct:.1f}% used)")
    print()
    return used_mb, avail_mb

In [3]:
# check RAM

check_ram("before import")


[before import] RAM Usage
  [░░░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :     105 MB
  Available     :    7081 MB
  Total system  :    7792 MB  (1.3% used)



(104.72265625, 7080.5078125)

## **Step 1: Import Data**

In [4]:
raw = mne.io.read_raw_eeglab(
    "sub-010_ses-01_task-experiment_run-01_eeg.set",
    preload=True,
    eog=['HEOGL', 'HEOGR', 'VEOGU', 'VEOGL']
)
 
print(f"Loaded: {len(raw.ch_names)} channels @ {raw.info['sfreq']} Hz")
check_ram("after import data")

Reading /home/cindy/BHSFinal/sub10_sess1.fdt
Reading 0 ... 4958827  =      0.000 ...  4842.604 secs...


/tmp/ipykernel_493/3292213545.py:1: RuntimeWarning: Estimated head radius (0.0 cm) is below the 3rd percentile for infant head size. Check if the montage_units argument is correct (the default is "mm", but your channel positions may be in different units).
  raw = mne.io.read_raw_eeglab(


Loaded: 128 channels @ 1024.0 Hz

[after import data] RAM Usage
  [███████████████████░░░░░░░░░░░]
  This notebook :    5094 MB
  Available     :    2110 MB
  Total system  :    7792 MB  (65.4% used)



/tmp/ipykernel_493/3292213545.py:1: RuntimeWarning: Not setting positions of 4 eog channels found in montage:
['HEOGR', 'HEOGL', 'VEOGU', 'VEOGL']
Consider setting the channel types to be of EEG/sEEG/ECoG/DBS/fNIRS using inst.set_channel_types before calling inst.set_montage, or omit these channels when creating your montage.
  raw = mne.io.read_raw_eeglab(


(5093.984375, 2109.8671875)

*The RuntimeWarning simply stated that the head radius estimated from the dataset's channel location is too small (which equals to 0.0cm).*

*Which is even smaller than an infant head. So it assume that there could be an error from the dataset*

*But I suggest that it could happen because of the datasets don't include the channel position, so it will be solved on the set montage part.*

## **Step 2: Downsample**

In [5]:
ORIGINAL_SFREQ = raw.info['sfreq']
TARGET_SFREQ   = 250   #change the sampling frequenct to reduce the dataset size

print(f"\nDownsampling {ORIGINAL_SFREQ} Hz -> {TARGET_SFREQ} Hz")
#MNE applies an anti-aliasing filter automatically during resample"

raw.resample(TARGET_SFREQ, npad='auto')
gc.collect()
 
print(f"\nNew sampling rate: {raw.info['sfreq']} Hz")
print(f"New duration check: {raw.times[-1]:.1f}s") #unchanged

check_ram("after downsample")


Downsampling 1024.0 Hz -> 250 Hz

New sampling rate: 250.0 Hz
New duration check: 4842.6s

[after downsample] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1426 MB
  Available     :    5853 MB
  Total system  :    7792 MB  (18.3% used)



(1425.94140625, 5852.84765625)

In [6]:
sfreq = raw.info['sfreq']
nyquist = sfreq / 2
 
print(f"Current sampling rate : {sfreq} Hz")
print(f"Nyquist limit         : {nyquist} Hz")

Current sampling rate : 250.0 Hz
Nyquist limit         : 125.0 Hz


## **Step 3: Check for data information**

In [7]:
# Data basic information
raw.info

<Info | 8 non-empty values
 bads: []
 ch_names: Fp1, Fpz, Fp2, F7, F3, Fz, F4, F8, FC5, FC1, FC2, FC6, M1, T7, ...
 chs: 124 EEG, 4 EOG
 custom_ref_applied: False
 dig: 127 items (3 Cardinal, 124 EEG)
 highpass: 0.0 Hz
 lowpass: 125.0 Hz
 meas_date: unspecified
 nchan: 128
 projs: []
 sfreq: 250.0 Hz
>

In [8]:
# Total channels 
len(raw.ch_names)

128

In [9]:
# Channel names
print(raw.ch_names)

['Fp1', 'Fpz', 'Fp2', 'F7', 'F3', 'Fz', 'F4', 'F8', 'FC5', 'FC1', 'FC2', 'FC6', 'M1', 'T7', 'C3', 'Cz', 'C4', 'T8', 'M2', 'CP5', 'CP1', 'CP2', 'CP6', 'P7', 'P3', 'Pz', 'P4', 'P8', 'POz', 'O1', 'O2', 'HEOGR', 'AF7', 'AF3', 'AF4', 'AF8', 'F5', 'F1', 'F2', 'F6', 'FC3', 'FCz', 'FC4', 'C5', 'C1', 'C2', 'C6', 'CP3', 'CP4', 'P5', 'P1', 'P2', 'P6', 'HEOGL', 'PO3', 'PO4', 'VEOGU', 'FT7', 'FT8', 'TP7', 'TP8', 'PO7', 'PO8', 'VEOGL', 'FT9', 'FT10', 'TPP9h', 'TPP10h', 'PO9', 'PO10', 'P9', 'P10', 'AFF1', 'AFz', 'AFF2', 'FFC5h', 'FFC3h', 'FFC4h', 'FFC6h', 'FCC5h', 'FCC3h', 'FCC4h', 'FCC6h', 'CCP5h', 'CCP3h', 'CCP4h', 'CCP6h', 'CPP5h', 'CPP3h', 'CPP4h', 'CPP6h', 'PPO1', 'PPO2', 'I1', 'Iz', 'I2', 'AFp3h', 'AFp4h', 'AFF5h', 'AFF6h', 'FFT7h', 'FFC1h', 'FFC2h', 'FFT8h', 'FTT9h', 'FTT7h', 'FCC1h', 'FCC2h', 'FTT8h', 'FTT10h', 'TTP7h', 'CCP1h', 'CCP2h', 'TTP8h', 'TPP7h', 'CPP1h', 'CPP2h', 'TPP8h', 'PPO9h', 'PPO5h', 'PPO6h', 'PPO10h', 'POO9h', 'POO3h', 'POO4h', 'POO10h', 'OI1h', 'OI2h']


In [10]:
# PSD orginal plots

initial_psd = raw.compute_psd(fmin=0, fmax=100, n_fft=2048, picks='eeg')
 
fig = initial_psd.plot(spatial_colors=True, show=False)
fig.savefig("EEG_plots/01_psd_raw.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/01_psd_raw.png")
 
del initial_psd #to save ram
gc.collect()
check_ram("after PSD")

Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).


/tmp/ipykernel_493/1837782227.py:5: RuntimeWarning: Channel locations not available. Disabling spatial colors.
  fig = initial_psd.plot(spatial_colors=True, show=False)


-> Saved: EEG_plots/01_psd_raw.png

[after PSD] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1433 MB
  Available     :    5835 MB
  Total system  :    7792 MB  (18.4% used)



(1433.23828125, 5834.84765625)

In [11]:
# Plot raw data

fig = raw.plot(duration=1, n_channels=5, start=500, show=False)
fig.savefig("EEG_plots/02_raw_snippet.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/02_raw_snippet.png")
 
gc.collect()
check_ram("after raw.plot()")

Using matplotlib as 2D backend.
-> Saved: EEG_plots/02_raw_snippet.png

[after raw.plot()] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1463 MB
  Available     :    5801 MB
  Total system  :    7792 MB  (18.8% used)



(1462.65234375, 5801.375)

## **Step 4: Set Montage**

In [12]:
# set montage using 10-10 system
montage = mne.channels.make_standard_montage("standard_1005")
raw.set_montage(montage, on_missing="ignore")

<RawEEGLAB | sub10_sess1.fdt, 128 x 1210651 (4842.6 s), ~1.15 GiB, data loaded>

In [13]:
# Check whether montage is successfully set
print(raw.get_montage().get_positions()['ch_pos']['Fp1']) #result shows yes

[-0.03090259  0.11458518  0.02786657]


In [14]:
# Plot 2D
fig = raw.plot_sensors(ch_type='eeg', show_names=True, show=False)
fig.savefig("EEG_plots/03_sensor_positions.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("Saved: EEG_plots/03_sensor_positions.png")

Saved: EEG_plots/03_sensor_positions.png


In [15]:
# Check whether all channels fit into standard_1005
montage = mne.channels.make_standard_montage("standard_1005")
montage_names = set(montage.ch_names)
 
eeg_names = [raw.ch_names[i] for i in mne.pick_types(raw.info, eeg=True)]
eeg_names_set = set(eeg_names)
 
matched    = eeg_names_set & montage_names
unmatched  = eeg_names_set - montage_names
 
print(f"Total EEG channels from the dataset : {len(eeg_names)}")
print(f"Matched to standard_1005            : {len(matched)}")
print(f"NOT matched (no position!)          : {len(unmatched)}")
 
if unmatched:
    print(f"\nUnmatched channels:")
    for ch in sorted(unmatched):
        print(f"  - {ch}")
else:
    print("\nAll EEG channels matched successfully. Montage is complete.")

Total EEG channels from the dataset : 124
Matched to standard_1005            : 124
NOT matched (no position!)          : 0

All EEG channels matched successfully. Montage is complete.


## **Step 5: Detecting Bad Channels**

#### *(Part 1) Initial Check*

In [16]:
# bad channel functions

def mark_bad_channels(
    raw,
    chunk_duration=20,
    n_chunks=4, #separate the recordinng into 4 chunks to reduce ram load
    min_vote_frac=0.5, #only identify as bad if bads appear at more than half of the chunks
    flat_thresh_uv=0.5, #if amplitude less than 0.5, consider as flats
    noisy_thresh_sd=3.0, #above 3 std consider as noisy channel
    corr_thresh=0.4, #below 0.4 consider poor correlation channel
    n_neighbours=4, #choose 4 channels around to compare
    verbose=True #print each chunk's progress
):
    
    check_ram("start of mark_bad_channels")
 
    # create a copy to detect bad channels
    raw_filt = raw.copy()
    raw_filt.filter(l_freq=1.0, h_freq=45.0, picks='eeg', verbose=False)
 
    sfreq     = raw_filt.info['sfreq']
    total_dur = raw_filt.times[-1]
    margin    = min(20, total_dur * 0.05)
    safe_dur  = total_dur - 2 * margin
 
    chunk_starts = [margin + i * (safe_dur / n_chunks) for i in range(n_chunks)]
    chunk_starts = [t for t in chunk_starts if t + chunk_duration <= total_dur]
 
    eeg_picks = mne.pick_types(raw_filt.info, eeg=True)
    eeg_names = [raw_filt.ch_names[i] for i in eeg_picks]
    n_eeg     = len(eeg_picks)
 
    if verbose:
        print(f"Detecting bad channels on {n_eeg} EEG channels")
        print(f"Recording: {total_dur:.0f}s | Testing {len(chunk_starts)} x {chunk_duration}s chunks")
        print("(Temporary 1-45Hz filter applied to a COPY -- original raw is untouched)\n")
 
    # Bad channel detection from neighbor channels
    pos2d = np.array([raw_filt.info['chs'][i]['loc'][:2] for i in eeg_picks]) #Get the 2D positions of each EEG channel
    neighbours = {}
    for i in range(n_eeg):
        dists = np.linalg.norm(pos2d - pos2d[i], axis=1) #count the distance to each of the channels
        dists[i] = np.inf #ignore its own channel
        neighbours[i] = np.argsort(dists)[:n_neighbours] #choose the nearest neighbours

    # Create the vote counters
    vote_flat  = {n: 0 for n in eeg_names}
    vote_noisy = {n: 0 for n in eeg_names}
    vote_corr  = {n: 0 for n in eeg_names}
 
    # Main part: bad channels detection
    for chunk_i, t_start in enumerate(chunk_starts):
        t_end   = t_start + chunk_duration
        s_start = int(t_start * sfreq) # convert seconds to samples
        s_stop  = int(t_end * sfreq) # convert seconds to samples
 
        if verbose:
            print(f"Chunk {chunk_i+1}/{len(chunk_starts)}: {t_start:.0f}-{t_end:.0f}s")
 
        data_uv = raw_filt.get_data(picks=eeg_picks, start=s_start, stop=s_stop) * 1e6 #convert to microvolt
        stds    = data_uv.std(axis=1)
        med_std = np.median(stds)
 
        # check for flats, std, neighbours, correlation
        for i, name in enumerate(eeg_names):
            if stds[i] < flat_thresh_uv:
                vote_flat[name] += 1
                if verbose:
                    print(f"  FLAT   : {name} (std={stds[i]:.3f} uV)")
 
            if stds[i] > med_std + noisy_thresh_sd * stds.std():
                vote_noisy[name] += 1
                if verbose:
                    print(f"  NOISY  : {name} (std={stds[i]:.1f} uV, median={med_std:.1f})")
 
            nn_idx    = neighbours[i]
            corrs     = [np.corrcoef(data_uv[i], data_uv[j])[0, 1] for j in nn_idx] #compute correlation from the neighbor's list
            mean_corr = np.mean(corrs)
            if mean_corr < corr_thresh:
                vote_corr[name] += 1
                if verbose:
                    print(f"  LOWCORR: {name} (mean r={mean_corr:.3f})")
 
        del data_uv, stds #free memory, delete variables that are no longer needed 
        gc.collect() #free the deleted objects to release more memory
 
    # Voting summary
    n_run     = len(chunk_starts) #how many chunks were analyzed
    threshold = max(1, int(np.ceil(min_vote_frac * n_run))) #bad channels should at least appeared at 2 of the 4 chunks 
 
    bads = []
    for name in eeg_names:
        if (vote_flat[name] >= threshold or
            vote_noisy[name] >= threshold or
            vote_corr[name] >= threshold):
            bads.append(name)
 
    if verbose:
        print(f"\n-- Results (threshold >= {threshold}/{n_run} chunks) --")
        for name in bads:
            reasons = []
            if vote_flat[name]  >= threshold: reasons.append(f"flat({vote_flat[name]})")
            if vote_noisy[name] >= threshold: reasons.append(f"noisy({vote_noisy[name]})")
            if vote_corr[name]  >= threshold: reasons.append(f"low_corr({vote_corr[name]})")
            print(f"  BAD: {name:<12s} reasons: {', '.join(reasons)}")
        if not bads:
            print("  No bad channels detected.")
 
    vote_detail = {'flat': vote_flat, 'noisy': vote_noisy, 'corr': vote_corr,
                   'threshold': threshold, 'n_chunks': n_run}
 
    # Clean up the temporary filtered copy -- original raw was never touched
    del raw_filt
    gc.collect()
    check_ram("end of mark_bad_channels")
 
    return bads, vote_detail


In [17]:
bads, vote_detail = mark_bad_channels(raw)


[start of mark_bad_channels] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1471 MB
  Available     :    5785 MB
  Total system  :    7792 MB  (18.9% used)

Detecting bad channels on 124 EEG channels
Recording: 4843s | Testing 4 x 20s chunks
(Temporary 1-45Hz filter applied to a COPY -- original raw is untouched)

Chunk 1/4: 20-40s
  LOWCORR: C5 (mean r=-0.975)
  NOISY  : CP3 (std=13775.5 uV, median=811.0)
  LOWCORR: CP3 (mean r=-0.993)
  LOWCORR: P1 (mean r=0.064)
  LOWCORR: PO3 (mean r=-0.869)
  NOISY  : FCC3h (std=13448.8 uV, median=811.0)
  LOWCORR: FCC3h (mean r=-0.795)
  NOISY  : CPP3h (std=39960.0 uV, median=811.0)
  LOWCORR: CPP3h (mean r=-0.475)
Chunk 2/4: 1221-1241s
  LOWCORR: C5 (mean r=-0.183)
  LOWCORR: CP3 (mean r=-0.441)
  LOWCORR: P1 (mean r=0.372)
  LOWCORR: PO3 (mean r=0.017)
  LOWCORR: FCC3h (mean r=-0.436)
  NOISY  : CPP3h (std=1590.8 uV, median=13.5)
  LOWCORR: CPP3h (mean r=-0.272)
  LOWCORR: PPO1 (mean r=0.318)
  LOWCORR: TTP7h (mean r=0.292)


*Results show chunks 3 and 4 are having lots of low correlation and flats which led to suspicious data collection.*

*I assume that there should be flat channels or cutoff in the eeg raw data.*

**Flats and abnormal channel detection**

In [19]:
# Check for flats

eeg_picks = mne.pick_types(raw.info, eeg=True)
sfreq = raw.info['sfreq']
total_dur = raw.times[-1]
 
print(f"Total recording duration: {total_dur:.1f}s")
print(f"Sampling rate: {sfreq} Hz")
print()
 
# Sample at many points across the recording to find where data goes flat
test_points = np.linspace(10, total_dur - 10, 20)
 
print(f"{'Time (s)':>10s}  {'Mean abs amplitude (uV)':>25s}  {'Status':>10s}")
print("-" * 50)
 
for t in test_points:
    s_idx = int(t * sfreq)
    s_end = s_idx + int(1 * sfreq)   # 1 second sample
    data = raw.get_data(picks=eeg_picks, start=s_idx, stop=s_end) * 1e6
    mean_abs = np.abs(data).mean()
    status = "FLAT/DEAD" if mean_abs < 0.01 else "active"
    print(f"{t:>10.1f}  {mean_abs:>25.4f}  {status:>10s}")
 
print()
print("Total annotations:", len(raw.annotations))

Total recording duration: 4842.6s
Sampling rate: 250.0 Hz

  Time (s)    Mean abs amplitude (uV)      Status
--------------------------------------------------
      10.0                  6415.1640      active
     263.8                  5766.5090      active
     517.6                  5797.2128      active
     771.5                  5811.1812      active
    1025.3                  5792.6762      active
    1279.1                  5609.3886      active
    1532.9                     0.0362      active
    1786.7                     0.0102      active
    2040.6                     0.0073   FLAT/DEAD
    2294.4                     0.0062   FLAT/DEAD
    2548.2                     0.0056   FLAT/DEAD
    2802.0                     0.0052   FLAT/DEAD
    3055.9                     0.0050   FLAT/DEAD
    3309.7                     0.0048   FLAT/DEAD
    3563.5                     0.0046   FLAT/DEAD
    3817.3                     0.0046   FLAT/DEAD
    4071.1                     0.0045   

In [20]:
# When the EEG signal breaks

eeg_picks = mne.pick_types(raw.info, eeg=True)
sfreq = raw.info['sfreq']

print("Zooming in on the transition zone (1279s - 1533s)...\n")
 
test_points = np.linspace(1279, 1533, 30)
print(f"{'Time (s)':>10s}  {'Mean abs amplitude (uV)':>25s}  {'Status':>10s}")
print("-" * 50)
 
cutoff_time = None
for t in test_points:
    s_idx = int(t * sfreq)
    s_end = s_idx + int(1 * sfreq)
    data = raw.get_data(picks=eeg_picks, start=s_idx, stop=s_end) * 1e6
    mean_abs = np.abs(data).mean()
    status = "active" if mean_abs > 1.0 else "FLAT/DEAD"
    print(f"{t:>10.1f}  {mean_abs:>25.4f}  {status:>10s}")
    if status == "active":
        cutoff_time = t
 
print(f"\nApproximate cutoff: real signal ends around t={cutoff_time:.1f}s")
 

events, event_id = mne.events_from_annotations(raw, verbose=False)
events_sec = events[:, 0] / sfreq
n_events_before_cutoff = (events_sec < cutoff_time).sum()
n_events_after_cutoff  = (events_sec >= cutoff_time).sum()
 
print(f"\nEvents BEFORE cutoff (t < {cutoff_time:.0f}s): {n_events_before_cutoff}")
print(f"Events AFTER cutoff  (t >= {cutoff_time:.0f}s): {n_events_after_cutoff}")

Zooming in on the transition zone (1279s - 1533s)...

  Time (s)    Mean abs amplitude (uV)      Status
--------------------------------------------------
    1279.0                  5608.8707      active
    1287.8                  5546.2198      active
    1296.5                  5556.3280      active
    1305.3                  5705.3448      active
    1314.0                  5642.0021      active
    1322.8                  5646.3282      active
    1331.6                  5624.0898      active
    1340.3                  5628.4644      active
    1349.1                  5613.4978      active
    1357.8                  5640.0687      active
    1366.6                  5630.3964      active
    1375.3                  5641.1879      active
    1384.1                  5629.1240      active
    1392.9                  5664.7661      active
    1401.6                  5887.8970      active
    1410.4                  5739.8324      active
    1419.1                  5978.1029      ac

In [23]:
# Cropped data first, then check the number in each trials

check_ram("before crop")
 
CUTOFF_TIME = 1463.0   
SAFETY_MARGIN = 2.0    # seconds of buffer before the dead zone starts
 
crop_end = CUTOFF_TIME - SAFETY_MARGIN
 
print(f"Original duration : {raw.times[-1]:.1f}s")
print(f"Cropping to        : 0 - {crop_end:.1f}s")
 
raw.crop(tmin=0, tmax=crop_end)
 
print(f"New duration       : {raw.times[-1]:.1f}s")
 
gc.collect()
check_ram("after crop")
 
# Re-check events after crop to confirm nothing was lost incorrectly
events, event_id = mne.events_from_annotations(raw, verbose=False)
print(f"\nEvents remaining after crop: {len(events)}")
for name, code in event_id.items():
    count = (events[:, 2] == code).sum()
    print(f"  {name:<20s}: {count}")



[before crop] RAM Usage
  [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :     646 MB
  Available     :    6609 MB
  Total system  :    7792 MB  (8.3% used)

Original duration : 1461.0s
Cropping to        : 0 - 1461.0s
New duration       : 1461.0s

[after crop] RAM Usage
  [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :     646 MB
  Available     :    6609 MB
  Total system  :    7792 MB  (8.3% used)


Events remaining after crop: 283
  0, Imagination_a_flower_high_10###my_stream_name: 2
  0, Imagination_a_flower_high_17###my_stream_name: 1
  0, Imagination_a_flower_high_23###my_stream_name: 1
  0, Imagination_a_flower_high_5###my_stream_name: 1
  0, Imagination_a_flower_low_10###my_stream_name: 2
  0, Imagination_a_flower_low_25###my_stream_name: 2
  0, Imagination_a_flower_normal_11###my_stream_name: 1
  0, Imagination_a_flower_normal_16###my_stream_name: 1
  0, Imagination_a_flower_normal_17###my_stream_name: 1
  0, Imagination_a_flower_normal_21###my_stream_name: 2
  0, Imagi

In [24]:
# Check trials from each stimulus

events, event_id = mne.events_from_annotations(raw) #this is the cropped data
 
print(f"Cropped recording duration: {raw.times[-1]:.1f}s")
print(f"Total events remaining in cropped data: {len(events)}\n")

clean_event_id = {}

for k, v in event_id.items():
    clean_name = k.split(", ")[-1]      # remove "0, "
    clean_name = clean_name.replace("###my_stream_name", "")
    clean_event_id[clean_name] = v


# Each stimulus events
image_events = [k for k in clean_event_id.keys() if k.startswith("Perception_image")]
text_events  = [k for k in clean_event_id.keys() if k.startswith("Perception_t")]
audio_events = [k for k in clean_event_id.keys() if k.startswith("Perceptiona")]


# Count trials per condition 
def count_trials(event_names, label):
    codes = [clean_event_id[n] for n in event_names]
    count = np.isin(events[:, 2], codes).sum()
    print(f"{label:<10s}: {count:>5d} trials   (event types: {event_names})")
    return count
 
print("\n" + "="*65)
print("USABLE TRIAL COUNTS PER CONDITION (within cropped/valid data)")
print("="*65)
 
img_count = count_trials(image_events, "Image")
txt_count = count_trials(text_events,  "Text")
aud_count = count_trials(audio_events, "Audio")
 
# Summary
print("\n" + "="*65)
print("BALANCE SUMMARY")
print("="*65)
 
counts = {"Image": img_count, "Text": txt_count, "Audio": aud_count}
min_count = min(counts.values())
max_count = max(counts.values())
 
for name, c in counts.items():
    print(f"  {name:<10s}: {c}")
 
if min_count == 0:
    print("\n🔴 At least one condition has ZERO trials in the valid window.")
    print("   That condition cannot be used for this subject's comparison.")
elif max_count / max(min_count, 1) > 2:
    print(f"\n⚠️  Imbalanced conditions (largest is {max_count/min_count:.1f}x the smallest).")
    print("   Keep this in mind when comparing ERP amplitudes across conditions --")
    print("   the condition with fewer trials will have a noisier average.")
else:
    print("\n✓ Conditions are reasonably balanced.")


Used Annotations descriptions: [np.str_('0, Imagination_a_flower_high_10###my_stream_name'), np.str_('0, Imagination_a_flower_high_17###my_stream_name'), np.str_('0, Imagination_a_flower_high_23###my_stream_name'), np.str_('0, Imagination_a_flower_high_5###my_stream_name'), np.str_('0, Imagination_a_flower_low_10###my_stream_name'), np.str_('0, Imagination_a_flower_low_25###my_stream_name'), np.str_('0, Imagination_a_flower_normal_11###my_stream_name'), np.str_('0, Imagination_a_flower_normal_16###my_stream_name'), np.str_('0, Imagination_a_flower_normal_17###my_stream_name'), np.str_('0, Imagination_a_flower_normal_21###my_stream_name'), np.str_('0, Imagination_a_flower_normal_23###my_stream_name'), np.str_('0, Imagination_a_flower_normal_5###my_stream_name'), np.str_('0, Imagination_a_flower_normal_6###my_stream_name'), np.str_('0, Imagination_a_guitar_high_14###my_stream_name'), np.str_('0, Imagination_a_guitar_high_25###my_stream_name'), np.str_('0, Imagination_a_guitar_high_5###my

#### *(Part 2) Real Check after cropped data*

In [25]:
bads, vote_detail = mark_bad_channels(raw)
raw.info['bads'] = bads
print(f"Synced: raw.info['bads'] = {raw.info['bads']}") #sync the bad channels to the original cropped data


[start of mark_bad_channels] RAM Usage
  [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :     646 MB
  Available     :    6551 MB
  Total system  :    7792 MB  (8.3% used)

Detecting bad channels on 124 EEG channels
Recording: 1461s | Testing 4 x 20s chunks
(Temporary 1-45Hz filter applied to a COPY -- original raw is untouched)

Chunk 1/4: 20-40s
  LOWCORR: C5 (mean r=-0.975)
  NOISY  : CP3 (std=13775.5 uV, median=811.0)
  LOWCORR: CP3 (mean r=-0.993)
  LOWCORR: P1 (mean r=0.064)
  LOWCORR: PO3 (mean r=-0.869)
  NOISY  : FCC3h (std=13448.8 uV, median=811.0)
  LOWCORR: FCC3h (mean r=-0.795)
  NOISY  : CPP3h (std=39960.0 uV, median=811.0)
  LOWCORR: CPP3h (mean r=-0.475)
Chunk 2/4: 375-395s
  LOWCORR: CP1 (mean r=0.389)
  LOWCORR: C5 (mean r=-0.132)
  LOWCORR: CP3 (mean r=-0.437)
  LOWCORR: P1 (mean r=0.286)
  LOWCORR: PO3 (mean r=0.142)
  LOWCORR: FCC3h (mean r=-0.324)
  NOISY  : CPP3h (std=1061.8 uV, median=14.4)
  LOWCORR: CPP3h (mean r=-0.258)
Chunk 3/4: 730-750s
  LOWCORR: CP1 (

In [27]:
# Plot the bad channels

eeg_picks = mne.pick_types(raw.info, eeg=True, exclude = [])
eeg_names = [raw.ch_names[i] for i in eeg_picks]
 
# Get 2D positions for all EEG channels
pos2d = np.array([raw.info['chs'][i]['loc'][:2] for i in eeg_picks])
 
fig, ax = plt.subplots(figsize=(8, 8))
 
# Plot all good channels in light gray
for i, name in enumerate(eeg_names):
    color = '#f87c7c' if name in bads else '#7c9ef8'
    size  = 120 if name in bads else 40
    zorder = 5 if name in bads else 2
    ax.scatter(pos2d[i, 0], pos2d[i, 1], c=color, s=size, zorder=zorder,
               edgecolors='black' if name in bads else 'none', linewidths=1)
    if name in bads:
        ax.annotate(name, (pos2d[i, 0], pos2d[i, 1]),
                    textcoords="offset points", xytext=(6, 6),
                    fontsize=9, fontweight='bold', color='#a83232')
 
# Draw head outline (approximate circle)
head_radius = np.max(np.linalg.norm(pos2d, axis=1)) * 1.05
circle = plt.Circle((0, 0), head_radius, fill=False, color='gray', lw=1.5)
ax.add_patch(circle)
 
# Nose marker (pointing up, +y direction)
ax.plot([0, 0], [head_radius, head_radius * 1.1], color='gray', lw=1.5)
 
ax.set_aspect('equal')
ax.set_title(f"Bad Channel Locations (n={len(bads)})\nRed = flagged bad, Blue = good",
             fontsize=12, fontweight='bold')
ax.axis('off')
 
fig.tight_layout()
fig.savefig("EEG_plots/04c_bad_channel_topography.png", dpi=120, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/04c_bad_channel_topography.png")


-> Saved: EEG_plots/04c_bad_channel_topography.png


## **Step 6: Re-reference**

In [28]:
raw_ref = raw.set_eeg_reference('average')

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.


## **Step 7: ICA**

In [29]:
check_ram("before ICA")


[before ICA] RAM Usage
  [██░░░░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :     649 MB
  Available     :    6548 MB
  Total system  :    7792 MB  (8.3% used)



(648.72265625, 6547.609375)

In [30]:
# Create two copies of raw_ref

raw_for_ica = raw_ref.copy() # for ica filtered
raw_for_analysis = raw_ref.copy() # for ica analysis

In [31]:
# ICA filtered

# Filtered with an aggressive high-pass
raw_for_ica = raw_for_ica.copy().filter(l_freq=1.0, h_freq=40.0)

ica = mne.preprocessing.ICA(
    n_components=40,
    method='fastica',
    random_state=97,
    max_iter=800
)

# Fit ICA on EEG channels only
picks_ica = mne.pick_types(raw_for_ica.info, eeg=True, eog=False, exclude="bads")
ica.fit(raw_for_ica, picks=picks_ica)

gc.collect()
check_ram("after ICA fit")

Filtering raw data in 1 contiguous segment
Setting up band-pass filter from 1 - 40 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 40.00 Hz
- Upper transition bandwidth: 10.00 Hz (-6 dB cutoff frequency: 45.00 Hz)
- Filter length: 825 samples (3.300 s)

Fitting ICA to data using 115 channels (please be patient, this may take a while)
Selecting by number: 40 components
Fitting ICA took 32.5s.

[after ICA fit] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1398 MB
  Available     :    5810 MB
  Total system  :    7792 MB  (17.9% used)



(1398.42578125, 5809.6640625)

In [33]:
# Remove blink components

eog_idx, eog_scores = ica.find_bads_eog(raw_for_ica)
ica.exclude = eog_idx.copy()
print(f"\nComponents marked for removal: {ica.exclude}")

Using EOG channels: HEOGR, HEOGL, VEOGU, VEOGL
... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2500 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff

In [34]:
# Appply ICA

raw_clean = ica.apply(raw_for_analysis)

Applying ICA to Raw instance
    Transforming to ICA space (40 components)
    Zeroing out 4 ICA components
    Projecting back using 115 PCA components


## **Step 8: Filtering**

#### **1. Notch Filter**

In [35]:
sfreq = raw_clean.info['sfreq']
nyquist = sfreq / 2
 
print(f"Current sampling rate : {sfreq} Hz")
print(f"Nyquist limit         : {nyquist} Hz")
print()

Current sampling rate : 250.0 Hz
Nyquist limit         : 125.0 Hz



In [36]:
notches = np.array([50, 100])  #not exceed the nyquist limit
raw_clean.notch_filter(
    freqs=notches,
    picks='eeg',
    filter_length='auto',
    phase='zero-double',
    fir_design='firwin',
    n_jobs=1            # keep at 1 for RAM safety
)
print(f"Notch filter applied at: {notches} Hz")

Filtering raw data in 1 contiguous segment
Setting up band-stop filter

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower transition bandwidth: 0.50 Hz
- Upper transition bandwidth: 0.50 Hz
- Filter length: 1651 samples (6.604 s)

Notch filter applied at: [ 50 100] Hz


In [37]:
# Plot PSD after notch
psd = raw_clean.compute_psd(fmin=0, fmax=100, n_fft=2048, picks='eeg')
fig = psd.plot(spatial_colors=True, show=False)
fig.savefig("EEG_plots/05_psd_after_notch.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/05_psd_after_notch.png")
 
del psd
gc.collect()
check_ram("after notch filter")

Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).
-> Saved: EEG_plots/05_psd_after_notch.png

[after notch filter] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1419 MB
  Available     :    5727 MB
  Total system  :    7792 MB  (18.2% used)



(1419.2890625, 5727.0)

#### **2. High Pass Filter**

In [38]:
raw_clean.filter(l_freq=1.0, h_freq=None, picks='eeg', n_jobs=1)
print("High-pass filter applied at 1Hz")

Filtering raw data in 1 contiguous segment
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 825 samples (3.300 s)

High-pass filter applied at 1Hz


In [39]:
# Plot PSD after high-pass
filtered_psd = raw_clean.compute_psd(fmin=0, fmax=100, n_fft=2048, picks='eeg')
fig = filtered_psd.plot(spatial_colors=True, show=False)
fig.savefig("EEG_plots/06_psd_after_highpass.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/06_psd_after_highpass.png")
 
del filtered_psd
gc.collect()
check_ram("after high-pass filter")

Effective window size : 8.192 (s)
Plotting power spectral density (dB=True).
-> Saved: EEG_plots/06_psd_after_highpass.png

[after high-pass filter] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1421 MB
  Available     :    5725 MB
  Total system  :    7792 MB  (18.2% used)



(1421.484375, 5725.44140625)

## **Step 9: Epoching**

In [40]:
# Get events from raw_clean

events, event_id = mne.events_from_annotations(raw_clean)

Used Annotations descriptions: [np.str_('0, Imagination_a_flower_high_10###my_stream_name'), np.str_('0, Imagination_a_flower_high_17###my_stream_name'), np.str_('0, Imagination_a_flower_high_23###my_stream_name'), np.str_('0, Imagination_a_flower_high_5###my_stream_name'), np.str_('0, Imagination_a_flower_low_10###my_stream_name'), np.str_('0, Imagination_a_flower_low_25###my_stream_name'), np.str_('0, Imagination_a_flower_normal_11###my_stream_name'), np.str_('0, Imagination_a_flower_normal_16###my_stream_name'), np.str_('0, Imagination_a_flower_normal_17###my_stream_name'), np.str_('0, Imagination_a_flower_normal_21###my_stream_name'), np.str_('0, Imagination_a_flower_normal_23###my_stream_name'), np.str_('0, Imagination_a_flower_normal_5###my_stream_name'), np.str_('0, Imagination_a_flower_normal_6###my_stream_name'), np.str_('0, Imagination_a_guitar_high_14###my_stream_name'), np.str_('0, Imagination_a_guitar_high_25###my_stream_name'), np.str_('0, Imagination_a_guitar_high_5###my

In [41]:
# Clean the events name

clean_event_id = {}

for k, v in event_id.items():
    clean_name = k.split(", ")[-1]      # remove "0, "
    clean_name = clean_name.replace("###my_stream_name", "")
    clean_event_id[clean_name] = v

# print(clean_event_id)

In [42]:
# Create modalities group

# Image
image_events = [
    k for k in clean_event_id.keys()
    if k.startswith("Perception_image")
]


# Text
text_events = [
    k for k in clean_event_id.keys()
    if k.startswith("Perception_t")
]


# Audio
audio_events = [
    k for k in clean_event_id.keys()
    if k.startswith("Perceptiona")
]

print("Image:", len(image_events))
print("Text:", len(text_events))
print("Audio:", len(audio_events))

print("Image:", image_events)
print("Text:", text_events)
print("Audio:", audio_events)

Image: 9
Text: 15
Audio: 36
Image: ['Perception_image_flower_c', 'Perception_image_flower_m', 'Perception_image_flower_s', 'Perception_image_guitar_c', 'Perception_image_guitar_m', 'Perception_image_guitar_s', 'Perception_image_penguin_c', 'Perception_image_penguin_m', 'Perception_image_penguin_s']
Text: ['Perception_t_flower_1', 'Perception_t_flower_2', 'Perception_t_flower_3', 'Perception_t_flower_4', 'Perception_t_flower_5', 'Perception_t_guitar_1', 'Perception_t_guitar_2', 'Perception_t_guitar_3', 'Perception_t_guitar_4', 'Perception_t_guitar_5', 'Perception_t_penguin_1', 'Perception_t_penguin_2', 'Perception_t_penguin_3', 'Perception_t_penguin_4', 'Perception_t_penguin_5']
Audio: ['Perceptiona_flower_high_10', 'Perceptiona_flower_high_17', 'Perceptiona_flower_high_23', 'Perceptiona_flower_high_5', 'Perceptiona_flower_low_10', 'Perceptiona_flower_low_25', 'Perceptiona_flower_normal_11', 'Perceptiona_flower_normal_16', 'Perceptiona_flower_normal_17', 'Perceptiona_flower_normal_21', 

In [43]:
# Convert the new groups into event_id

image_event_id = {k: clean_event_id[k] for k in image_events}
text_event_id  = {k: clean_event_id[k] for k in text_events}
audio_event_id = {k: clean_event_id[k] for k in audio_events}

print("Image: ", image_event_id)
print("Text: ", text_event_id)
print("Audio: ", audio_event_id)

Image:  {'Perception_image_flower_c': 62, 'Perception_image_flower_m': 63, 'Perception_image_flower_s': 64, 'Perception_image_guitar_c': 65, 'Perception_image_guitar_m': 66, 'Perception_image_guitar_s': 67, 'Perception_image_penguin_c': 68, 'Perception_image_penguin_m': 69, 'Perception_image_penguin_s': 70}
Text:  {'Perception_t_flower_1': 71, 'Perception_t_flower_2': 72, 'Perception_t_flower_3': 73, 'Perception_t_flower_4': 74, 'Perception_t_flower_5': 75, 'Perception_t_guitar_1': 76, 'Perception_t_guitar_2': 77, 'Perception_t_guitar_3': 78, 'Perception_t_guitar_4': 79, 'Perception_t_guitar_5': 80, 'Perception_t_penguin_1': 81, 'Perception_t_penguin_2': 82, 'Perception_t_penguin_3': 83, 'Perception_t_penguin_4': 84, 'Perception_t_penguin_5': 85}
Audio:  {'Perceptiona_flower_high_10': 86, 'Perceptiona_flower_high_17': 87, 'Perceptiona_flower_high_23': 88, 'Perceptiona_flower_high_5': 89, 'Perceptiona_flower_low_10': 90, 'Perceptiona_flower_low_25': 91, 'Perceptiona_flower_normal_11': 9

In [44]:
# Combine all event_id

all_event_id = {}

all_event_id.update(image_event_id)
all_event_id.update(text_event_id)
all_event_id.update(audio_event_id)

print(all_event_id)

{'Perception_image_flower_c': 62, 'Perception_image_flower_m': 63, 'Perception_image_flower_s': 64, 'Perception_image_guitar_c': 65, 'Perception_image_guitar_m': 66, 'Perception_image_guitar_s': 67, 'Perception_image_penguin_c': 68, 'Perception_image_penguin_m': 69, 'Perception_image_penguin_s': 70, 'Perception_t_flower_1': 71, 'Perception_t_flower_2': 72, 'Perception_t_flower_3': 73, 'Perception_t_flower_4': 74, 'Perception_t_flower_5': 75, 'Perception_t_guitar_1': 76, 'Perception_t_guitar_2': 77, 'Perception_t_guitar_3': 78, 'Perception_t_guitar_4': 79, 'Perception_t_guitar_5': 80, 'Perception_t_penguin_1': 81, 'Perception_t_penguin_2': 82, 'Perception_t_penguin_3': 83, 'Perception_t_penguin_4': 84, 'Perception_t_penguin_5': 85, 'Perceptiona_flower_high_10': 86, 'Perceptiona_flower_high_17': 87, 'Perceptiona_flower_high_23': 88, 'Perceptiona_flower_high_5': 89, 'Perceptiona_flower_low_10': 90, 'Perceptiona_flower_low_25': 91, 'Perceptiona_flower_normal_11': 92, 'Perceptiona_flower_no

#### **Create Epoch**

In [46]:
epochs = mne.Epochs(
    raw_clean,
    events,
    event_id=all_event_id,
    tmin=-0.2,
    tmax=2.0,
    baseline=(-0.2, 0),
    picks='eeg',
    preload=True,
    detrend=None,
    reject=dict(eeg=200e-6),
    flat=dict(eeg=1e-6), # also reject suspiciously flat/dead epochs
    reject_by_annotation=True
)

print(f"\n{epochs}")
 
gc.collect()
check_ram("after epoching")

Not setting metadata
137 matching events found
Applying baseline correction (mode: mean)
0 projection items activated
Using data from preloaded Raw for 137 events and 551 original time points ...
    Rejecting  epoch based on EEG : ['PPO9h']
    Rejecting  epoch based on EEG : ['FTT9h']
    Rejecting  epoch based on EEG : ['OI2h']
    Rejecting  epoch based on EEG : ['AF7', 'FT10', 'FTT10h']
    Rejecting  epoch based on EEG : ['T8', 'FTT9h']
    Rejecting  epoch based on EEG : ['PO9', 'I1', 'POO9h']
    Rejecting  epoch based on EEG : ['Fp2']
    Rejecting  epoch based on EEG : ['O1', 'AF7', 'I1', 'POO9h']
    Rejecting  epoch based on EEG : ['O2', 'I2', 'PPO9h', 'POO10h', 'OI2h']
    Rejecting  epoch based on EEG : ['O1', 'PO9', 'I1', 'POO9h']
    Rejecting  epoch based on EEG : ['O1', 'O2', 'I1', 'I2', 'POO9h', 'POO10h', 'OI2h']
    Rejecting  epoch based on EEG : ['O1', 'PO9', 'I1', 'POO9h']
    Rejecting  epoch based on EEG : ['PO9']
    Rejecting  epoch based on EEG : ['F8', 'F6'

(1468.453125, 5678.125)

In [47]:
# split epochs by modality

epochs_text = epochs[text_events]
epochs_image = epochs[image_events]
epochs_audio = epochs[audio_events]

In [ ]:
#5) Save epochs to disk
#epochs.save("sub-010_clean-epo.fif", overwrite=True)
#print("-> Saved: sub-010_clean-epo.fif")
 
#gc.collect()
#check_ram("end of epoching")

## **Step 10: ERP Averanging**

In [48]:
evoked_image = epochs_image.average()
evoked_text  = epochs_text.average()
evoked_audio = epochs_audio.average()
 
print(f"Image ERP: averaged from {evoked_image.nave} trials")
print(f"Text ERP : averaged from {evoked_text.nave} trials")
print(f"Audio ERP: averaged from {evoked_audio.nave} trials")
 
conditions = [
    ('Image', evoked_image, '#7c9ef8'),
    ('Text',  evoked_text,  '#f87c7c'),
    ('Audio', evoked_audio, '#7cf8b8')
]

Image ERP: averaged from 31 trials
Text ERP : averaged from 30 trials
Audio ERP: averaged from 29 trials


In [49]:
# Butterfly plots

fig, axes = plt.subplots(3, 1, figsize=(12, 10), sharex=True)
for ax, (name, evoked, color) in zip(axes, conditions):
    evoked.plot(axes=ax, spatial_colors=True, show=False, time_unit='ms')
    ax.set_title(f"ERP -- {name} stimulus (n={evoked.nave} trials)",
                 color=color, fontweight='bold')
    ax.axvline(0, color='black', lw=1.2, ls='--')
fig.suptitle("ERP Butterfly Plots by Condition", fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig("EEG_plots/07_erp_butterfly.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/07_erp_butterfly.png")
 
gc.collect()

/tmp/ipykernel_493/231521012.py:10: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  fig.tight_layout()


-> Saved: EEG_plots/07_erp_butterfly.png


14495

In [50]:
# Single channel overlay

channels_to_check = ['Pz', 'Oz', 'Cz']
 
for ch in channels_to_check:
    if ch not in evoked_image.ch_names:
        print(f"Skipping {ch} -- not in channel list")
        continue
 
    ch_idx = evoked_image.ch_names.index(ch)
    times_ms = evoked_image.times * 1000
 
    fig, ax = plt.subplots(figsize=(11, 5))
    for name, evoked, color in conditions:
        ax.plot(times_ms, evoked.data[ch_idx] * 1e6, label=f"{name} (n={evoked.nave})",
                 color=color, lw=2)
    ax.axvline(0, color='black', lw=1.2, ls='--', alpha=0.7, label='Stimulus onset')
    ax.axhline(0, color='gray', lw=0.8, ls=':')
    ax.set_xlabel("Time (ms)")
    ax.set_ylabel("Amplitude (µV)")
    ax.set_title(f"ERP at {ch}: Image vs Text vs Audio", fontsize=13, fontweight='bold')
    ax.legend(fontsize=10)
    ax.set_xlim([-200, 1000])
    fig.tight_layout()
    fig.savefig(f"EEG_plots/08_erp_{ch}_comparison.png", dpi=100, bbox_inches='tight')
    plt.close(fig)
    print(f"-> Saved: EEG_plots/08_erp_{ch}_comparison.png")
 
gc.collect()
check_ram("after single-channel plots")


-> Saved: EEG_plots/08_erp_Pz_comparison.png
Skipping Oz -- not in channel list
-> Saved: EEG_plots/08_erp_Cz_comparison.png

[after single-channel plots] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1528 MB
  Available     :    5621 MB
  Total system  :    7792 MB  (19.6% used)



(1527.70703125, 5620.58203125)

In [51]:
# Topomap comparison at ERP latencies

timepoints = [0.1, 0.17, 0.3, 0.4]
 
fig, axes = plt.subplots(3, len(timepoints), figsize=(16, 9))
for row, (name, evoked, color) in enumerate(conditions):
    for col, tp in enumerate(timepoints):
        ev_crop = evoked.copy().crop(tmin=tp - 0.01, tmax=tp + 0.01)
        mne.viz.plot_topomap(
            ev_crop.data.mean(axis=1),
            evoked.info,
            axes=axes[row, col],
            show=False,
            cmap='RdBu_r'
        )
        if row == 0:
            axes[row, col].set_title(f"{int(tp*1000)} ms", fontsize=12, fontweight='bold')
        if col == 0:
            axes[row, col].set_ylabel(name, fontsize=12, color=color, fontweight='bold')
 
fig.suptitle("Brain Topographies at Key Latencies\n(Image / Text / Audio)",
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig("EEG_plots/09_topomaps_by_condition.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/09_topomaps_by_condition.png")
 
gc.collect()
check_ram("after topomaps")


-> Saved: EEG_plots/09_topomaps_by_condition.png

[after topomaps] RAM Usage
  [█████░░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1541 MB
  Available     :    5608 MB
  Total system  :    7792 MB  (19.8% used)



(1541.20703125, 5607.83203125)

In [52]:
# Waveform + topomap

for name, evoked, color in conditions:
    try:
        fig = evoked.plot_joint(
            times=[0.1, 0.17, 0.3, 0.4],
            title=f"ERP Joint Plot -- {name} (n={evoked.nave})",
            show=False
        )
        fig.savefig(f"EEG_plots/10_joint_{name.lower()}.png", dpi=100, bbox_inches='tight')
        plt.close(fig)
        print(f"-> Saved: EEG_plots/10_joint_{name.lower()}.png")
    except Exception as e:
        print(f"Joint plot skipped for {name}: {e}")
 
gc.collect()

No projector specified for this dataset. Please consider the method self.add_proj.
-> Saved: EEG_plots/10_joint_image.png
No projector specified for this dataset. Please consider the method self.add_proj.
-> Saved: EEG_plots/10_joint_text.png
No projector specified for this dataset. Please consider the method self.add_proj.
-> Saved: EEG_plots/10_joint_audio.png


53577

In [54]:
# Global Field Power (GFP)

"""
GFP = sqrt(mean(signal^2 across all channels)) at each timepoint.
It collapses all 115 channels into ONE number per timepoint --
basically "how strong is the overall brain response right now".
 
Peaks in GFP = moments where many channels are responding strongly
together. This is the cleanest way to spot WHEN peaks happen without
115 lines fighting for attention.
"""
 
fig, ax = plt.subplots(figsize=(12, 6))
 
for name, evoked, color in conditions:
    data = evoked.data  # (n_channels, n_times) in Volts
    gfp = np.sqrt((data ** 2).mean(axis=0)) * 1e6   # -> µV
    times_ms = evoked.times * 1000
    ax.plot(times_ms, gfp, label=f"{name} (n={evoked.nave})", color=color, lw=2.2)
 
ax.axvline(0, color='black', lw=1.2, ls='--', alpha=0.7, label='Stimulus onset')
ax.set_xlabel("Time (ms)")
ax.set_ylabel("Global Field Power (µV)")
ax.set_title("Global Field Power: Image vs Text vs Audio\n(overall signal strength across all channels)",
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_xlim([-200, 1000])
ax.grid(alpha=0.2)
fig.tight_layout()
fig.savefig("EEG_plots/11_gfp_comparison.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/11_gfp_comparison.png")
 
gc.collect()


-> Saved: EEG_plots/11_gfp_comparison.png


3930

In [55]:
# Targeted channel subset

"""
Pick a small, meaningful set of channels covering key brain regions:
  Occipital (O1, O2, Oz)       -- visual processing, expect strong for image
  Parietal  (P3, P4, Pz)       -- attention, P300
  Central   (C3, C4, Cz)       -- general/motor
  Frontal   (F3, F4, Fz)       -- cognitive control
 
This gives you regional coverage without 115 overlapping lines.
"""
 
subset_channels = ['O1', 'O2', 'Oz', 'P3', 'P4', 'Pz', 'C3', 'C4', 'Cz', 'F3', 'F4', 'Fz']
subset_channels = [ch for ch in subset_channels if ch in evoked_image.ch_names]
 
print(f"\nUsing {len(subset_channels)} channels: {subset_channels}")
 
fig, axes = plt.subplots(4, 3, figsize=(16, 13), sharex=True, sharey=True)
axes = axes.flatten()
 
for idx, ch in enumerate(subset_channels):
    ax = axes[idx]
    ch_idx = evoked_image.ch_names.index(ch)
    times_ms = evoked_image.times * 1000
 
    for name, evoked, color in conditions:
        ax.plot(times_ms, evoked.data[ch_idx] * 1e6, color=color, lw=1.3, label=name)
 
    ax.axvline(0, color='black', lw=0.8, ls='--', alpha=0.5)
    ax.axhline(0, color='gray', lw=0.5, ls=':')
    ax.set_title(ch, fontsize=11, fontweight='bold')
    ax.set_xlim([-200, 1000])
 
    if idx == 0:
        ax.legend(fontsize=8, loc='upper right')
 
# Hide unused subplots if subset_channels < 12
for idx in range(len(subset_channels), len(axes)):
    axes[idx].axis('off')
 
fig.supxlabel("Time (ms)", fontsize=12)
fig.supylabel("Amplitude (µV)", fontsize=12)
fig.suptitle("ERP by Region: Image vs Text vs Audio\n(12 representative channels)",
             fontsize=14, fontweight='bold')
fig.tight_layout()
fig.savefig("EEG_plots/12_erp_regional_grid.png", dpi=100, bbox_inches='tight')
plt.close(fig)
print("-> Saved: EEG_plots/12_erp_regional_grid.png")
 
gc.collect()
check_ram("after clean ERP views")



Using 11 channels: ['O1', 'O2', 'P3', 'P4', 'Pz', 'C3', 'C4', 'Cz', 'F3', 'F4', 'Fz']
-> Saved: EEG_plots/12_erp_regional_grid.png

[after clean ERP views] RAM Usage
  [██████░░░░░░░░░░░░░░░░░░░░░░░░]
  This notebook :    1560 MB
  Available     :    5586 MB
  Total system  :    7792 MB  (20.0% used)



(1560.47265625, 5585.58984375)